### **Task 5: Fine-Tuning BERT for POS Tagging & Chunking**

**Objective**

The objective of this assignment is to build and fine-tune a transformer-based model for token classification tasks in Natural Language Processing. The model will perform Part-of-Speech (POS) tagging and chunking to identify grammatical roles of words and phrase structures in sentences. A pretrained model such as DistilBERT will be trained on the CoNLL-2003 dataset and the complete pipeline will include preprocessing, training, evaluation and inference.


### **1: Dataset Selection**

**Install Required Libraries**

In [1]:
!pip uninstall -y datasets
!pip install datasets==2.18.0 transformers seqeval evaluate accelerate -q

Found existing installation: datasets 2.18.0
Uninstalling datasets-2.18.0:
  Successfully uninstalled datasets-2.18.0


In [2]:
!pip install evaluate seqeval -q

**Import Libraries**

In [3]:
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

import evaluate

**Load Dataset**

In [4]:
dataset = load_dataset("conll2003")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1461: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

**Identify POS Labels**

In [5]:
label_list = dataset["train"].features["pos_tags"].feature.names

print("POS Labels:")
print(label_list)

POS Labels:
['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']


In [6]:
label_list = dataset["train"].features["chunk_tags"].feature.names

print("Chunk Labels:")
print(label_list)

Chunk Labels:
['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


## **2. Data Preprocessing**

**Load Tokenizer**

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")

**Tokenization and Label Alignment Function**

In [8]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["chunk_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

**Apply Tokenization**

In [ ]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

## **3. Model Setup**

**Label Mapping**

In [10]:
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

**Load Model**

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

## **4. Model Training**

In [12]:
small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(100))
small_val = tokenized_dataset["validation"].shuffle(seed=42).select(range(30))

**Define Training Arguments**

In [13]:
training_args = TrainingArguments(
    output_dir="./chunk_model",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01
)

**Define Evaluation MetricsTrainer**

In [14]:
metric = evaluate.load("seqeval")

In [15]:
def compute_metrics(eval_preds):

    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=2)

    true_predictions = [
        [label_list[p] for (p,l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p,l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

**Create Trainer**

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics
)

**Train the Model**

In [17]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=25, training_loss=2.7039337158203125, metrics={'train_runtime': 362.252, 'train_samples_per_second': 0.276, 'train_steps_per_second': 0.069, 'total_flos': 13070270976000.0, 'train_loss': 2.7039337158203125, 'epoch': 1.0})

### **5. Model Evaluation**

In [18]:
results = trainer.evaluate()

print("Evaluation Results")
print("Precision:", results["eval_precision"])
print("Recall:", results["eval_recall"])
print("F1 Score:", results["eval_f1"])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Evaluation Results
Precision: 0.15789473684210525
Recall: 0.15306122448979592
F1 Score: 0.15544041450777202


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## **6. Inference (Prediction on Custom Sentence)**

In [19]:
def predict_chunk(sentence):

    tokens = sentence.split()

    inputs = tokenizer(
        tokens,
        return_tensors="pt",
        is_split_into_words=True
    )

    outputs = model(**inputs)

    predictions = outputs.logits.argmax(dim=2)

    predicted_labels = [
        label_list[p.item()]
        for p in predictions[0]
    ]

    return list(zip(tokens, predicted_labels[1:len(tokens)+1]))

**Test the Model**

In [20]:
sentence = "John works at Google in California"
prediction = predict_chunk(sentence)
for word, tag in prediction:
    print(word, "→", tag)

John → I-NP
works → I-NP
at → B-NP
Google → I-NP
in → I-NP
California → I-NP


## **7. Comparison Between POS Tagging and Chunking**

Part-of-Speech (POS) Tagging and
Chunking are two important tasks in Natural Language Processing used for analyzing the grammatical structure of sentences.

POS tagging assigns a grammatical category to each individual word in a sentence. These categories include nouns, verbs, adjectives, adverbs, pronouns and other parts of speech. For example, in the sentence “John works at Google”, the word John is tagged as a proper noun, works as a verb, and Google as a proper noun. POS tagging focuses on identifying the role of each word separately.

Chunking, also known as shallow parsing, goes one step further by grouping words together into meaningful phrases such as noun phrases (NP), verb phrases (VP) and prepositional phrases (PP). Instead of labeling individual words only, chunking identifies the structure of phrases within a sentence. For example, in the sentence “John works at Google”, chunking would identify “John” as a noun phrase, “works” as a verb phrase, and “at Google” as a prepositional phrase.

The main difference between POS tagging and chunking is that POS tagging works at the word level, while chunking works at the phrase level. POS tagging is generally easier because it assigns a label to each word independently. Chunking is slightly more complex because it must understand how words combine to form phrases.

## **Task 8: Report / Blog**


In this assignment, a transformer-based model was fine-tuned to perform token classification tasks such as POS tagging and chunking. The goal was to build a system that can identify the grammatical role of words and detect phrase-level structures in a sentence. A pretrained transformer model, **DistilBERT**, was used because it is efficient and performs well for sequence labeling tasks.

The dataset used for this task was the **CoNLL-2003 dataset**, which contains annotated sentences with tokens, POS tags, chunk tags and named entity labels. In this project, chunk tags were used to train a token classification model that predicts phrase boundaries such as noun phrases, verb phrases and prepositional phrases.

One of the main challenges faced during this task was handling tokenization and label alignment. Transformer tokenizers sometimes split words into smaller subwords. Because of this, labels must be carefully aligned with the correct tokens while ignoring special tokens such as `[CLS]` and `[SEP]`. This was handled by assigning the value **-100** to tokens that should not be considered during training.

Another challenge was managing computational resources. Training large transformer models like **BERT** can consume significant memory and processing time. To solve this issue, a smaller model and a reduced subset of the dataset were used, which allowed the model to train efficiently while still demonstrating the token classification pipeline.

The results show that transformer models are highly effective for sequence labeling tasks. Even with a smaller dataset, the model was able to learn patterns and predict chunk labels for new sentences. This demonstrates the power of pretrained language models for natural language understanding tasks.

In conclusion, this project provided practical experience in building a token classification system using transformer models. The workflow included dataset preparation, tokenization, label alignment, model training, evaluation, and inference. These steps form the foundation for many advanced NLP applications such as named entity recognition, information extraction, and text analysis.
